In [ ]:
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# default: Load the model on the available device(s)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "/data/lib/huggingface/Qwen3-VL-2B-Instruct/", dtype="auto", device_map="auto"
)


# huggingface-cli download --resume-download Qwen/Qwen3-32B --local-dir Qwen3-32B
# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen3VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen3-VL-2B-Instruct",
#     dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]


The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


In [7]:
messages2 = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "image.png",
            },
            {"type": "text", "text": "这可能是几倍体，给出原因."},
        ],
    }
]

In [12]:
# Preparation for inference
inputs = processor.apply_chat_template(
    messages2,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)
inputs = inputs.to(model.device)

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=1024)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

['根据您提供的信息，我们可以分析该样本的基因组特征来判断其可能的倍性（即染色体组数）。\n\n### 1. 关键数据解读\n\n首先，我们从表格中提取关键数据：\n\n- **样本名称**: `taixian`\n- **Kmer**: 17\n- **深度 (Depth)**: 101\n- **n_kmer**: 41,221,639,234\n- **Genome_size(M)**: 408.14\n- **Revised Genome_size(M)**: 401.16\n- **Heterozygous_rate(%)**: 0.32\n- **Repeat_rate(%)**: 55.04\n\n### 2. 分析基因组大小和重复率\n\n- **基因组大小**: 408.14 Mbp (兆碱基对)。这个值是原始基因组大小，经过修正后为 401.16 Mbp。\n- **重复率**: 55.04%。这是一个非常高的重复率，表明基因组中存在大量的重复序列。\n- **异源性率**: 0.32%。这个值非常低，表明基因组中存在少量的异源序列（即来自不同物种的序列）。\n\n### 3. 评估倍性\n\n#### 3.1. 比较基因组大小\n- **原始基因组大小**: 408.14 Mbp\n- **修正后基因组大小**: 401.16 Mbp\n- **差异**: 408.14 - 401.16 = 6.98 Mbp\n\n这个差异非常小，表明基因组的大小在进化过程中没有发生显著变化，因此可以排除基因组大小的显著变化是倍性的主要因素。\n\n#### 3.2. 分析重复率\n- **重复率**: 55.04%。这个值非常高，表明基因组中存在大量的重复序列。\n- **重复序列**: 重复序列是基因组中常见的特征，通常与基因组的复杂性和进化历史有关。\n\n#### 3.3. 分析异源性率\n- **异源性率**: 0.32%。这个值非常低，表明基因组中存在少量的异源序列。\n- **异源序列**: 异源序列通常与基因组的复杂性和进化历史有关。\n\n### 4. 结论\n\n综合以上分析，我们可以得出以下结论：\n\n- **基因组大小**: 408.14 Mbp，修正后为 401.16 Mbp，差异很小，表明基因组大小没有显著变化。\n- **重复率

In [13]:
print(output_text[0])

根据您提供的信息，我们可以分析该样本的基因组特征来判断其可能的倍性（即染色体组数）。

### 1. 关键数据解读

首先，我们从表格中提取关键数据：

- **样本名称**: `taixian`
- **Kmer**: 17
- **深度 (Depth)**: 101
- **n_kmer**: 41,221,639,234
- **Genome_size(M)**: 408.14
- **Revised Genome_size(M)**: 401.16
- **Heterozygous_rate(%)**: 0.32
- **Repeat_rate(%)**: 55.04

### 2. 分析基因组大小和重复率

- **基因组大小**: 408.14 Mbp (兆碱基对)。这个值是原始基因组大小，经过修正后为 401.16 Mbp。
- **重复率**: 55.04%。这是一个非常高的重复率，表明基因组中存在大量的重复序列。
- **异源性率**: 0.32%。这个值非常低，表明基因组中存在少量的异源序列（即来自不同物种的序列）。

### 3. 评估倍性

#### 3.1. 比较基因组大小
- **原始基因组大小**: 408.14 Mbp
- **修正后基因组大小**: 401.16 Mbp
- **差异**: 408.14 - 401.16 = 6.98 Mbp

这个差异非常小，表明基因组的大小在进化过程中没有发生显著变化，因此可以排除基因组大小的显著变化是倍性的主要因素。

#### 3.2. 分析重复率
- **重复率**: 55.04%。这个值非常高，表明基因组中存在大量的重复序列。
- **重复序列**: 重复序列是基因组中常见的特征，通常与基因组的复杂性和进化历史有关。

#### 3.3. 分析异源性率
- **异源性率**: 0.32%。这个值非常低，表明基因组中存在少量的异源序列。
- **异源序列**: 异源序列通常与基因组的复杂性和进化历史有关。

### 4. 结论

综合以上分析，我们可以得出以下结论：

- **基因组大小**: 408.14 Mbp，修正后为 401.16 Mbp，差异很小，表明基因组大小没有显著变化。
- **重复率**: 55.04%。这个值非常高，表明基因组中存在大量的重复序列。
- **异源性率**